<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/Chapter_08A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_08A:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

### **Chapter 8: Statistical Inference for the Relationship Between Two Variables**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* **8.1 : Introduction**
* **8.2 : Relationship Between a Numerical Variable and a Binary Variable**
* **8.3 : Inference about the Relationship Between Two Binary Variables**
* 8.4 : Inference Regarding the Linear Relationship Between Two Numerical Variables
* 8.5 : Advanced

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 Key

from google.colab import userdata
import os

# Check if SAT1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA1403 key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 API key: {e}")
    print("Please set your STA1403 API key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 Key and toggle 'Notebook access' on")

If your STA1403 Key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 28
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your API is correctly installed.

## Load Data Sets

Run the next cell to load the data sets for this lesson.

In [ ]:
# @title Load Data Sets
import pandas as pd

# Load birthwt dataset
birthwt_df = pd.read_csv("https://biologicslab.co/STA1403/data/birthwt",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load platelet dataset
platelet_df = pd.read_csv("https://biologicslab.co/STA1403/data/platelet.csv",
                          sep = ',', na_values=[" ", "NA", "null", ""])

# Replace any na with median
#Pima_df['bmi'] = Pima_df['bmi'].fillna(Pima_df['bmi'].median())
#Pima_df.head()

# Load BodyTemperature.txt dataset
bodyTemp_df = pd.read_csv("https://biologicslab.co/STA1403/data/BodyTemperature.txt",
                         sep=' ', na_values=[" ", "NA", "null", ""])

# Load balance dataset
balance_df = pd.read_csv("https://biologicslab.co/STA1403/data/balance.csv", sep=',', na_values=[" ", "NA", "null", ""])

print("Datasets loaded sucessfully. ✅")

# **Chapter 8 Statistical Inference for the Relationship Between Two Variables**

## **8.1 Introduction**

In the previous two chapters, we discussed estimation and hypothesis testing regarding the population mean of a random variable. For instance, using sample data, we estimated the population mean of normal body temperature and tested the hypothesis that the population mean is less than the accepted value of $98.6^{\circ}$F. Often, however, the goal of scientific studies is to investigate the relationship between two (or more) variables. For example, we might be interested in investigating the relationship between gender and body temperature. In this chapter, we discuss estimation and hypothesis testing with respect to the relationship between two random variables. We start by discussing problems where we are investigating the relationship between one binary categorical variable (e.g., gender) and one numerical variable (e.g., body temperature). (More general situations where the categorical variable could take more than two possible values are discussed later.) We then discuss some statistical inference methods to examine relationship when both random variables are binary. Finally, we review situations where both random variables are numerical.

Throughout this chapter, we assume that the individuals from which we collect data are sampled randomly and independently from the population (unless stated otherwise). This will be the case if we use simple random sampling. Also, we assume that the sample size $n$ is large enough for the CLT to hold.

## **8.2 Relationship Between a Numerical Variable and a Binary Variable**

In this section, we discuss situations where we investigate possible relationship between a binary random variable and a numerical random variable. In these situations, the binary variable typically represents two different groups (e.g., smoking vs. non-smoking, male vs. female, cancer cells vs. normal cells) from the population or two different experimental conditions (e.g., treatment A vs. treatment B). In this section, we treat the binary variable as the explanatory variable in our analysis. The binary variable is also known as the **factor**. The numerical variable, on the other hand, is regarded as the response (target) variable (e.g., body temperature).

As a running example, suppose that we believe that gender and normal body temperature are related. That is, we believe that healthy men and women are different with respect to their body temperature. We can interpret this as difference in the distributions of body temperature between men and women. The two distributions (for men and women) can of course be different in many ways. (For example, one distribution could have higher variance than the other one.) For simplicity, we focus on the means of the two distributions. If we denote the population mean of body temperature $\mu_{1}$ for women and $\mu_{2}$ for men, the hypothesis that the two groups are different in terms of their body temperature can be specified as $H_{A}:\mu_{1} \neq \mu_{2}$. In other words, $H_{A}$ states that gender is an important factor with respect to body temperature and that the two characteristics are related. The corresponding null hypothesis is that the two means are equal: $H_{0}: \mu_{1} = \mu_{2}$.

-------------------

>In general, we can denote the means of the two groups as $\mu_{1}$ and $\mu_{2}$. The null hypothesis indicates that the population means are equal, $H_{0}:\mu_{1}=\mu_{2}$. In contrast, the alternative hypothesis is one of the following:
$$
\begin{array}{l}
H_{A}:\mu_{1}>\mu_{2} \text {  if we believe the mean for group 1 is greater} \\
\qquad\qquad\qquad \text {than the mean for group 2.} \\
H_{A}:\mu_{1}<\mu_{2} \text {  if we believe the mean for group 1 is less} \\
\qquad\qquad\qquad \text {than the mean for group 2.} \\
H_{A}:\mu_{1}\neq\mu_{2} \text {  if we believe the means are different} \\
\qquad\qquad\qquad  \text {but we do not specify which one is greater.} \\
\end{array}
$$
We can also express these hypotheses in terms of the difference in the means: $ H_{A}:\mu_{1}-\mu_{2}>0 $ , $ H_{A}:\mu_{1}-\mu_{2}<0 $ , or $ H_{A}:\mu_{1}-\mu_{2}\neq0 $. Then the corresponding null hypothesis is that there is no difference in the population means, $ H_{0}:\mu_{1}-\mu_{2}=0$.
More generally, we can express the null hypothesis in terms of the difference between the population means as $ H_{0}:\mu_{1}-\mu_{2}=\mu_{0}$. However, in most cases, $ \mu_{0}=0 $.

---------------

Previously, we used the sample mean $\bar{X}$ to perform statistical inference regarding the population mean $\mu$. To evaluate our hypothesis regarding the difference between two means, $\mu_{1}-\mu_{2}$, it is reasonable to choose the difference between the sample means, $\bar{X}_{1}-\bar{X}_{2}$, as our statistic. Here, $\bar{X}_{1}$ is the sample mean of the random variable of interest (e.g., body temperature) in the first group, and $\bar{X}_{2}$ is the sample mean in the second group. We use $\mu_{1 2}$ to denote the difference between the population means $\mu_{1}$ and $\mu_{2}$, and use $\bar{X}_{1 2}$ to denote the difference between the sample means $\bar{X}_{1}$ and $\bar{X}_{2}$.

$$
\begin{array}{l} \mu_{1 2} = \mu_{1} - \mu_{2}, \\
\bar {X}_{1 2} = \bar {X}_{1} - \bar {X} _{2}. \\
\end{array}
$$

A specific value of the test statistic $\bar{X}_{1 2}$ based on a sample of data is denoted $\bar{x}_{1 2}$ and calculated as

$$
\bar{x}_{1 2} = \bar {x}_{1} - \bar{x}_{2},
$$

where $\bar{x}_{1}$ and $\bar{x}_{2}$ are the observed sample means for group 1 and group 2, respectively. In this case, $\bar{x}_{1 2}$ is our point estimate for $\mu{1}-\mu_{2}$, the difference between population means.

For the above example, suppose that our sample includes $n_{1}=25$ women and $n_{2}=27$ men. The sample mean of body temperature is $\bar{x}_{1}=98.2$ for women and $\bar{x}_{2}=98.4$ for men. Then, our point estimate for the difference between population means is $\bar{x}_{1 2}=-0.2$.

As discussed before, point estimates do not reflect our uncertainty of our guess for unknown values (here, the difference between population means). To address this issue, we use interval estimates. Finding confidence intervals for the difference between two means is quite similar to steps we followed to find confidence intervals for one population mean. To start, we suppose that the sample variances for both groups are known. For the above example, we assume that $\sigma_{1}^{2}=0.8$ and $\sigma_{2}^{2}=1$.

Now, we need to find the sampling distribution of $\bar{X}_{1 2}$. By the Central Limit Theorem, the sampling distributions of $\bar{X}_{1}$ and $\bar{X}_{2}$ are approximately normal (exactly normal if the random variable itself is normally distributed) as follows:

$$
\begin{array}{l} \bar {X}_{1} \sim N \left(\mu_{1}, \sigma_{1}^{2} / n_{1}\right), \\
\bar {X}_{2} \sim N \left(\mu_{2}, \sigma_ {2}^{2} / n_{2}\right), \\
\end{array}
$$

where $ n_{1} $ and $ n_{2} $ are the number of observations, and $\sigma_{1}^{2}$ and $\sigma_{2}^{2}$ are the population variances for body temperature in group 1 and group 2, respectively.

The statistic $\bar{X}_{1 2}$ is the difference between the two normally distributed variables $\bar{X}_{1}$ and $\bar{X}_{2}$. As discussed in Sect. 5.8, if two random variables are normally distributed, their difference is also normally distributed with the mean equal to the difference of the means and variance equal to the _sum_ of the variances. Therefore, the sampling distribution of $\bar{X}_{1 2}$ is also approximately normal as follows:

$$
\bar {X}_{1 2} \sim N \left(\mu_{1} - \mu_{2}, \sigma_{1}^{2} / n_{1} + \sigma_{2}^{2} /n_{2}\right).
$$

That is, the sampling distribution of $\bar{X}_{1 2}$ is normal with mean $\mu_{1 2} = \mu_{1} - \mu_{2}$ and variance $\sigma_{1}^{2}/n_{1} + \sigma_{2}^{2}/n_{2}$ . We use $ SD_{1 2} $ to denote the standard deviation of the sampling distribution of $\bar{X}_{1 2}$,

$$
SD_{1 2} = \sqrt {\sigma_{1}^{2} / n_{1} + \sigma_{2}^{2} / n_{2}}.
$$

Therefore, we can write the sampling distribution of $\bar{X}_{1 2} $ as follows:

$$
\bar {X}_{1 2} \sim N \left(\mu_{1 2}, SD_{1 2}^{2}\right).
$$

For our example, the variance of the sampling distribution is $0.8/25 + 1/27 = 0.07$, and the standard deviation is $SD_{1 2} = \sqrt{0.07} = 0.26$.

As before, we can use our point estimate and the corresponding standard deviation to find confidence intervals. In this case, the confidence interval for $ \mu_{12} = \mu_{1} - \mu_{2} $ is obtained as follows:

$$
[\bar {x}_{1 2} - z_{\mathrm{crit}} \times SD_{1 2}, \bar{x}_{1 2} + z_{\mathrm {crit}} \times SD_{1 2} ],
$$

where $ z_{crit} $ is obtained for a given confidence level $c$ as before. For example, the 95% confidence interval for the difference between the population means of body temperature for women and men is
$$
[- 0.2 - 2 \times 0.26, - 0.2 + 2 \times 0.26 ] = [- 0.72, 0.32].
$$

Therefore, at $0.95$ confidence level, we believe that the true difference between the two means falls between $-0.72$ and $0.32$.

Note that the above confidence interval shows that the difference could be negative or positive. More specifically, the interval includes $0$, which is interpreted as no difference between the two means, i.e., no difference between women and men in terms of mean body temperature. Therefore, even though our point estimate for the difference between the means is negative (lower mean body temperature among women compared to men), our confidence interval shows that the true difference (i.e., between population means) is quite likely to be positive (i.e., higher mean body temperature among women compared to men). As a result, we cannot say with confidence that mean body temperature among women is lower than that of men, even though our point estimate indicates that. In what follows, we discuss this more formally in the context of hypothesis testing.

We now return to our hypothesis that $H_{A}:\mu_{1 2}\neq0$ (i.e., the difference between the two means is not zero) against the null hypothesis that $H_{0}:\mu_{1 2}=0$. To use $\bar{X}_{1 2}$ as a test statistic, we need to find its sampling distribution under the null hypothesis (i.e., its null distribution). We found that the sampling distribution of $\bar{X}_{1 2}$ is

$$
\bar {X}_{1 2} \sim N \left(\mu_{1 2}, SD_{1 2}^ {2}\right).
$$

If the null hypothesis is true, then $\mu_{1 2}=0$. Therefore, the null distribution of $\bar{X}_{1 2}$ is

$$
\bar {X}_{1 2} \sim N \left(0, SD_{1 2} ^ {2}\right).
$$

For the body temperature example, the null distribution of $\bar{X}_{1 2}$ is $N(0, 0.26^{2})$.

Because we can find the distribution of $\bar{X}_{1 2}$ under the null (even though it is an approximate distribution when the random variable is not normally distributed), we can use it as a test statistics to examine hypotheses regarding the difference between the means of two groups, $\mu_{1 2}$. As before, however, it is more common to standardize the test statistic by subtracting its mean (under the null) and dividing the result by its standard deviation. In this case, of course, the mean of the $ \bar{X}_{1 2} $ under the null hypothesis is zero. Therefore,

$$
Z = \frac {\bar {X}_{1 2}}{SD_{1 2}},
$$

where $Z$ is called the $z$-statistic, and it has the standard normal distribution: $Z \sim N(0,1)$. Similarly, we standardize the observed value of the test statistic, $\bar{x}_{1 2}$:

$$
z = \frac {\bar {x}_{1 2}}{SD_{1 2}}.
$$

We refer to $z$ as the $z$-score. For the body temperature example, the $z$-score is
$$
z = \frac {-0.2}{0.26} = -0.76.
$$

-------------------

>To test the null hypothesis $H_{0}:\mu_{12}=0$, we determine the $z$-score. Then, depending on the alternative hypothesis, we can calculate the _p_-value, which is the observed significance level, as:
$$
\begin{array}{l}
\text {if  } H_{A}: \mu_ {1 2} > 0, \quad p _ {\mathrm {obs}} = P (Z \geq z), \\
\text {if  } H_{A}: \mu_{1 2} < 0, \quad p_{\mathrm {obs}} = P (Z \leq z), \\
\text {if  } H_{A}: \mu_{1 2} \neq 0, \quad p_{\mathrm {obs}} = 2 \times P (Z \geq | z |). \\
\end{array}
$$
The above tail probabilities are obtained from the standard normal distribution. This hypothesis testing procedure is known as the **two-sample z-test**.

-----------------

For our example, $H_{A}:\mu_{12}\neq0$ and $z=-0.76$. Therefore, $p_{\mathrm{obs}}=2P(Z\geq|-0.76|)=2\times0.22=0.44$.

As before, we use the _p_-value to measure the amount of evidence against the null hypothesis. To decide whether we should reject the null hypothesis, we compare $p_{obs}$ with predefined significance levels (cutoffs) such as $0.01$, $0.05$ and $0.1$. For a given cutoff, we reject the null hypothesis and conclude that the result of our test is statistically significant if $p_{obs}$ is less than the cutoff.

For the body temperature example, $p_{obs} = 0.44$ is greater than the commonly used significance levels (e.g., $0.01$, $0.05$, and $0.1$). Therefore, the test result is not statistically significant, and we cannot reject the null hypothesis (which states that the population means for the two groups are the same) at these levels. That is, any observed difference could be due to chance alone. Recall that when we cannot reject the null hypothesis, our test remains inconclusive since our failure to reject the null could be either due to the fact that the null hypothesis is true, or it could be the case that the null hypothesis is false, but we do not have enough evidence to reject it.

### **_8.2.1 Two-Sample t-tests for Comparing the Means_**

So far, we have assumed that the population variances $\sigma_{1}^{2}$ and $\sigma_{2}^{2}$ for the two groups are known, so we could find the standard deviation, $SD_{12}$, of the sampling distribution of $\bar{X}_{1 2}$. In general, this is not a realistic assumption. In this section, we discuss statistical inference regarding population means for two groups where population variances $\sigma_{1}^{2}$ and $ \sigma_{2}^{2}$ are unknown. As before, we can use the sample variances $S_{1}^{2}$ and $S_{2}^{2}$ to estimate $\sigma_{1}^{2}$ and $ \sigma_{2}^{2}$, and take this additional source of uncertainty into account by using t-distributions instead of the standard normal distribution. We use $s_{1}^{2}$ and $s_{2}^{2}$ to denote the specific values of $S_{1}^{2}$ and $S_{2}^{2}$ based on the observed data. We regard $s_{1}^{2}$ and $s_{2}^{2}$ as our point estimates for population variances $\sigma_{1}^{2}$ and $\sigma_{2}^{2}$ and use them to estimate the standard deviation, $SD_{1 2}$, of the sampling distribution of $\bar{X}_{1 2}$.

Recall that the standard deviation of $\bar{X}_{1 2}$ is $SD_{1 2}=\sqrt{\sigma_{1}^{2}/n_{1}+\sigma_{2}^{2}/n_{2}}$. We refer to our estimate of this standard deviation as the _standard error_ of $\bar{X}_{1 2}$ and denote it as $SE_{1 2}$.

$$
SE_{1 2} = \sqrt {s_{1}^{2} / n_{1} + s_{2}^{2} / n_{2}}.
$$

For the body temperature example, suppose that the sample variances based on our sample of $n_{1}=25$ women and $n_{2}=27$ men are $s_{1}^{2}=1.1$ and $s_{2}^{2}=1.2$, respectively. Then the standard error of $\bar{X}_{1 2}$ is

$$
SE_{1 2} = \sqrt {1.1 / 25 + 1.2 / 27} = 0.30.
$$

Using the specific value of $\bar{X}_{1 2}$, which is denoted $\bar{x}_{1 2}$, as our point estimate for the difference between the two population means, $\mu_{1 2} = \mu_{1} - \mu_{2}$, along with the standard error $SE_{1 2}$ of $\bar{X}{1 2}$, we find confidence intervals for $ \mu_{1 2} $ as follows:

$$
[\bar {x}_{1 2} - t_{\mathrm {crit}} \times SE_{1 2}, \bar {x}_{1 2} + t_{\mathrm {crit}} \times SE_{1 2} ],
$$

where $t_{crit}$ is the _t_-critical value from a $t$-distribution for the desired confidence level $c$.

Previously, when we discussed statistical inference regarding one population mean with unknown variance, we used a $t$-distribution, whose degrees of freedom parameter $df$ was set to $n - 1$. When comparing the population means for two groups, the formula for finding the degrees of freedom is as follows:

$$
df = \frac {\left(s_{1}^{2} / n_{1} + s_{2}^{2} / n_{2}\right)^{2}}{\frac {1}{n_{1}- 1} \left(s_{1}^{2} / n_ {1}\right)^{2}+\frac {1}{n_{2} - 1} \left(s_{2}^{2} / n_{2}\right)^{2}}.
$$

For our example,
$$
df = \frac {(1.1 / 25 + 1.2 / 27) ^ {2}}{\frac {1}{25-1} (1.1 /25)^{2} + \frac {1}{27-1} (1.2 / 27)^{2}} = 49.9.
$$

Note that the degrees of freedom is not necessarily a whole number anymore as it is the case for inference regarding one population mean.

To find the corresponding $t_{crit}$, we follow similar steps as before. Suppose that we are interested in 95% confidence interval for $ \mu_{1 2}$ . We find $t_{crit}$ from the $t$-distribution with $df$ = $49.9$ degrees of freedom.

The formula for finding the degrees of freedom is slightly complex. As we will see later, for this type of hypothesis testing, we usually employ statistical software such as Python to perform this calculation as shown in Example 1.

The formula for finding the degrees of freedom is slightly complex. As we will see later, for this type of hypothesis testing, we usually employ statistical software such as R-Commander. Therefore, we rarely need to calculate the degrees of freedom manually. Alternatively, we could use a conservative approach and set df to $\min(n_{1}-1, n_{2}-1)$ , i.e., the smaller of $n_{1}-1$ and $ n_{2}-1$. This leads to slightly wider confidence intervals since it uses a slightly larger t-critical value. For the above example, we could set $df = \min(25-1, 27-1) = 24$ to be conservative. The corresponding $ t_{crit}$ for $0.95$ confidence level is $2.06$. This results in the following $95$% confidence interval:

$$
[- 0.2 - 2.06 \times 0.30, - 0.2 + 2.06 \times 0.30] = [- 0.82, 0.42 ],
$$

which is slightly wider than what we found previously based on a more exact calculation of the degrees of freedom.

For testing a hypothesis regarding $\mu_{1 2} = \mu_{1} - \mu_{2}$ when the population variances are unknown, we follow similar steps as above, but we use $SE_{1 2}$ instead of $SD_{1 2}$ and use the following _t_-statistic instead of the _z_-statistic to account for the additional source of uncertainty involved in estimating the population variances:

$$
T = \frac {\bar {X}_{1 2}}{\sqrt {S_{1}^{2} / n_{1} + S_{2}^{2} /n_{2}}},
$$

where $\bar{X}_{1 2}=\bar{X}_{1}-\bar{X}_{2}$ as before. Using the observed data, we obtain $\bar{x}_{1 2}=\bar{x}_{1}-\bar{x}_{2}$ as the observed value of $\bar{X}_{1 2}$. We also use the observed data to obtain $s_{1}$ and $s_{2}$ as the observed values of sample variances.

Then, we calculate the observed value of the test statistic $T$ as follows:

$$
\begin{array}{l} t = \frac {\bar {x} _ {1 2}}{\sqrt {s _ {1} ^ {2} / n _ {1} + s _ {2} ^ {2} / n _ {2}}} \\
= \frac {\bar {x}_{1 2}}{SE_{1 2}}, \\
\end{array} $$

which is called the **$t$-score**.

Depending on the alternative hypothesis, we calculate $ p_{obs} $ as
$$ \begin{array}{l} \mathrm {i f} H _ {A}: \mu_ {1 2} > 0, \quad p _ {\mathrm {o b s}} = P (T \geq t), \ \mathrm {i f} H _ {A}: \mu_ {1 2} < 0, \quad p _ {\mathrm {o b s}} = P (T \leq t), \ \mathrm {i f} H _ {A}: \mu_ {1 2} \neq 0, \quad p _ {\mathrm {o b s}} = 2 \times P (T \geq | t |), \ \end{array} $$
where $T$ has a $t$-distribution with the degrees of freedom obtained as above (Eq. 8.1). The hypothesis testing process is then called the two-sample $t$-test.
For the body temperature example,
$$ t = \frac {- 0 . 2}{0 . 3 0} = - 0. 6 7. $$



## Example 1: Find $t$-critical Value

The code in the cell below shows how to find the $t_{crit}$ value.

In [ ]:
# @title Example 1: Find t-critical Value

import numpy as np
from scipy import stats

# Define variables
n1 = 25
n2 = 27
s1 = 1.1
s2 = 1.2
mean_diff = -0.2
se_diff = 0.30
confidence_level = 0.95

# Calculate degrees of freedom using Welch's formula
df = ((s1**2 / n1) + (s2**2 / n2))**2 / (1/(n1-1) * \
            (s1**2 / n1)**2 + 1/(n2-1) * (s2**2 / n2)**2)

# Calculate critical t-value
alpha = 1 - confidence_level
t_critical = stats.t.ppf(1 - alpha/2, df)

# Calculate confidence interval
margin_of_error = t_critical * se_diff
ci_lower = mean_diff - margin_of_error
ci_upper = mean_diff + margin_of_error

print(f"Degrees of freedom: {df:.2f}")
print(f"Critical t-value: {t_critical:.2f}")
print(f"95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")



If the code is correct, you should see the following output:
```text
Degrees of freedom: 50.00
Critical t-value: 2.01
95% Confidence Interval: [-0.80, 0.40]
```

Therefore, at $0.95$ confidence level, we believe that the true difference between the two means falls between $-0.80$ and $0.40$.

The formula for finding the degrees of freedom is slightly complex. As we will see later, for this type of hypothesis testing, we usually employ statistical software such as Python. Therefore, we rarely need to calculate the degrees of freedom manually.

## **Exercise 1: Find $t$-critical Value**

In the cell below, write the code to find the $t_{crit}$ value.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below.
2. Change the data as shown in this code chunk:
```Python
# Define variables
n1 = 45
n2 = 27
s1 = 1.1
s2 = 3.2
mean_diff = -2.3
se_diff = 0.40
confidence_level = 0.95
```

In [ ]:
# @title Exercise 1: Find t-critical Value



If the code is correct, you should see the following output:
```text
Degrees of freedom: 29.73
Critical t-value: 2.04
95% Confidence Interval: [-3.12, -1.48]
```

Alternatively, we could use a conservative approach and set $df$ to $\min(n_{1}-1, n_{2}-1)$, i.e., the smaller of $n_{1}-1 $ and $n_{2}-1$. This leads to slightly wider confidence intervals since it uses a slightly larger $t$-critical value. For the above example, we could set $df = \min(25-1, 27-1) = 24$ to be conservative as shown in Example 2.

## Example 2: Find Conservative $t$-critical Value

The code in the cell below illustrates how to find a conservative value for $t_{crit}$ using the smaller $df$ value using the same data as in Example 1.

In [ ]:
# @title Example 2: Find Conservative t-critical Value

import numpy as np
from scipy import stats

# Define variables
n1 = 25
n2 = 27
s1 = 1.1
s2 = 1.2
mean_diff = -0.2
se_diff = 0.30
confidence_level = 0.95

# Conservative approach (using min(n1-1, n2-1))
df_con = min(n1-1, n2-1)
t_crit_con = stats.t.ppf(1 - alpha/2, df_con)

margin_of_error_con = t_crit_con * se_diff
ci_lower_con = mean_diff - margin_of_error_con
ci_upper_con = mean_diff + margin_of_error_con

print(f"\nConservative approach (df = {df_con}):")
print(f"Critical t-value: {t_crit_con:.2f}")
print(f"95% Confidence Interval: [{ci_lower_con:.2f}, {ci_upper_con:.2f}]")

If the code is correct, you should see the following output:
```text
Conservative approach (df = 24):
Critical t-value: 2.06
95% Confidence Interval: [-0.82, 0.42]
```

This results in the following 95% confidence interval:
$$ [ -0.2 - 2.06 \times 0.30, -0.2 + 2.0 6 \times 0.30] = [-0.82, 0.42], $$
which is slightly wider than what we found previously based on a more exact calculation of the degrees of freedom.

## **Exercise 2: Find Conservative $t$-critical Value**

In the cell below, write the code to find the conservative value for $t_{crit}$ using the smaller $df$ value using the same data as in **Exercise 1**.

In [ ]:
# @title Example 2: Find Conservative t-critical Value



If the code is correct, you should see the following output:
```text
Conservative approach (df = 26):
Critical t-value: 2.06
95% Confidence Interval: [-3.12, -1.48]
```


For testing a hypothesis regarding $\mu_{12} = \mu_{1} - \mu_{2}$ when the population variances are unknown, we follow similar steps as above, but we use $SE_{12}$ instead of $SD_{12}$ and use the following $t$-statistic instead of the $z$-statistic to account for the additional source of uncertainty involved in estimating the population variances:

$$
T = \frac {\bar {X}_{12}}{\sqrt {S_{1}^{2} / n_{1} + S_{2}^{2} / n_{2}}},
$$

where $ \bar{X}{12}=\bar{X}{1}-\bar{X}{2}$ as before. Using the observed data, we obtain $\bar{x}{12}=\bar{x}{1}-\bar{x}{2}$ as the observed value of $\bar{X}{12}$. We also use the observed data to obtain $s_{1}$ and $ s_{2}$ as the observed values of sample variances. Then, we calculate the observed value of the test statistic $T$ as follows:
$$
\begin{array}{l} t = \frac {\bar {x} _ {1 2}}{\sqrt {s _ {1} ^ {2} / n _ {1} + s _ {2} ^ {2} / n _ {2}}} \ = \frac {\bar {x} _ {1 2}}{S E _ {1 2}}, \\
\end{array}
$$
which is called the **$t$-score**.

Depending on the alternative hypothesis, we calculate $ p_{obs} $ as
$$ \begin{array}{l}
\text {if } H_{A}: \mu_{12} > 0, \quad p_{\mathrm {obs}} = P(T \geq t), \\
\text {if } H_{A}: \mu_{12} < 0, \quad p_{\mathrm {obs}} = P(T \leq t), \\
\text {if } H_{A}: \mu_{12} \neq 0, \quad p_{\mathrm {obs}} = 2\times P (T \geq | t |), \\
\end{array} $$
where $T$ has a $t$-distribution with the degrees of freedom obtained as above (Eq. 8.1). The hypothesis testing process is then called the **two-sample $t$-test**.

For the body temperature example,
$$t = \frac {- 0.2}{0.30} = -0.67.$$


## Example 3: Two-Sample t-Test

The code in the cell below performs a two-sample $t$-Test on body temperature.

In [ ]:
# @title Example 3: Two-Sample t-Test

import pandas as pd
import numpy as np
from scipy import stats

# Set seed
np.random.seed(42)  # For reproducible results

# Define variables
dataset = bodyTemp_df
col_name = "Gender"
x1 = "M"
x2 = "F"
sample_var = "Temperature"


# Set parameters
sample_1 = dataset[dataset[col_name] == x1]
sample_2 = dataset[dataset[col_name] == x2]

# Compute statistics --------------------------------
# number
n1 = len(sample_1)
n2 = len(sample_2)

# means
mean1 = sample_1[sample_var].mean()
mean2 = sample_2[sample_var].mean()

# standard deviations
s1 = sample_1[sample_var].std()
s2 = sample_2[sample_var].std()

# Calculate mean difference
mean_diff = mean1 - mean2

# Calculate standard error of the difference
se_diff = np.sqrt((s1**2 / n1) + (s2**2 / n2))

# Calculate degrees of freedom using Welch's formula
df = ((s1**2 / n1) + (s2**2 / n2))**2 / (1/(n1-1) * (s1**2 / n1)**2 + 1/(n2-1) * (s2**2 / n2)**2)

# Calculate t-score
t_score = mean_diff / se_diff

# Calculate p-value based on alternative hypothesis H_A: mu_12 != 0
# This is a two-tailed test
p_value = 2 * (1 - stats.t.cdf(abs(t_score), df))

# Display results
print("Two-Sample t-Test Results")
print("=" * 30)
print(f"Sample sizes: n1 = {n1}, n2 = {n2}")
print(f"Sample means: x̄₁ = {mean1:.2f}, x̄₂ = {mean2:.2f}")
print(f"Sample standard deviations: s₁ = {s1:.2f}, s₂ = {s2:.2f}")
print(f"Mean difference: x̄₁₂ = {mean_diff:.2f}")
print(f"Standard error: SE₁₂ = {se_diff:.2f}")
print(f"Degrees of freedom: df = {df:.2f}")
print(f"t-score: t = {t_score:.2f}")
print(f"p-value: p_obs = {p_value:.2f}")

If the code is correct, you should see the following output:
```text
Two-Sample t-Test Results
==============================
Sample sizes: n1 = 49, n2 = 51
Sample means: x̄₁ = 98.20, x̄₂ = 98.46
Sample standard deviations: s₁ = 1.05, s₂ = 0.85
Mean difference: x̄₁₂ = -0.26
Standard error: SE₁₂ = 0.19
Degrees of freedom: df = 92.27
t-score: t = -1.35
p-value: p_obs = 0.18

```

The output above shows resulting t-score, the degrees of freedom df, and the p-value. The $t$-score is $-1.35$, the degrees of freedom for the $t$-distribution (i.e., the null distribution) is $92.27$, and the corresponding $p$-value is $0.18$. Given the standard error, $SE_{12} = 0.19$, we are $95$% confident that the true value of $\mu_{12} = \mu_{1} - \mu_{2}$ is between $-0.64$ and $0.12$.

Because the $p$-value is $0.18$, we cannot reject the null hypothesis that $\mu_{12}=0$ at commonly used significance levels (0.01, 0.05, 0.1). We say the result is not statistically significant, and any observed difference could be due to chance alone.

As the second example, we consider an experiment where the amount of mean sway range (in millimeters) in the forward/backward plane and side/side plane were recorded for two groups of subjects, young and elderly, while taking part in a reaction time test $[35]$ . The data set includes $ n_{1}=9 $ elderly subjects and $n_{2}=8$ young subjects. Each subject was asked to stand barefoot on a “force platform” and maintain a stable upright position. Then, they were supposed to react as quickly as possible to an unpredictable noise by pressing a hand-held button. The noise was produced randomly. The platform automatically measured how much a subject swayed in millimeters in both the forward/backward and the side-to-side directions.

Determine whether there is a statistically significant difference in the amount of `forward_backward` sway between the `elderly` and `young` subjects in **Exercise 3A** below.

## **Exercise 3: Two-Sample t-Test**

Write the code in the cell below to determine whether there is a statistically significant difference in the amount of `forward_backward` sway between the elderly and young subjects in the DataFrame `balance_df` that was created at the start of this lesson.

**Code Hints:**

1. Copy-and-paste Example 3 into the cell below.
2. Change the variables to read as follows:
```Python
# Define variables
dataset = balance_df
col_name = "Age"
x1 = "elderly"
x2 = "young"
sample_var = "forward_backward"
```


In [ ]:
# @title Exercise 3: Two-Sample t-Test



If the code is correct, you should see the following output:
```text
Two-Sample t-Test Results
==============================
Sample sizes: n1 = 9, n2 = 8
Sample means: x̄₁ = 26.33, x̄₂ = 18.12
Sample standard deviations: s₁ = 9.77, s₂ = 4.09
Mean difference: x̄₁₂ = 8.21
Standard error: SE₁₂ = 3.56
Degrees of freedom: df = 10.97
t-score: t = 2.30
p-value: p_obs = 0.04
```

The $t$-score is t = $2.30$. Based on the $df = 10.97$, the standard error, $SE_{12} = 3.56$. Therefore, we are $95$% confident that the true value of $\mu_{12} = \mu_{1} - \mu_{2}$ is between 0.36 and 16.05. Note that the values in this range are all positive. More specifically, the value $0$ stated by the null hypothesis for $\mu_{12}$ is not included in this range. Why this is important will become clear when we investigate this more formally through hypothesis testing.

### **_8.2.2 Pooled t-test_**

When we used Python to perform two-sample $t$-tests, we assumed that the variances for the two groups were the same. Assuming $\sigma_{1}^{2}=\sigma_{2}^{2} $is not reasonable in general and should be avoided. Statisticians used to make this assumption for convenience when computer programs for statistical analysis were not available.

If we assume $\sigma_{1}^{2}=\sigma_{2}^{2}=\sigma^{2}$, we need to estimate only one (instead of two) variance parameter, $\sigma^{2}$, which is the common variance between the two groups. We estimate $\sigma^{2}$ using the **pooled sample variance**, $s_{p}^{2}$, as follows:
$$
s_{p}^{2} = \frac {\left(n_{1} - 1\right) s_{1}^{2} + \left(n_{2} - 1\right) s_{2}^{2}}{n_{1} + n_{2} - 2},
$$

where $n_{1}$ and $n_{2}$ are the sample sizes and $s_{1}$ and $s_{2}$ are sample variances for the two groups. Note that the pooled sample variance is in fact the weighted average of group-specific sample variances, where the $n_{1}-1$ and $n_{2}-1$ are the weights (i.e., the group with larger sample size is weighted higher).

To obtain the $t$-score, the $t$-critical value, and $p$-value, we follow a similar procedure as the standard two-sample $t$-test discussed above, but this time, we use $s_{p}^{2}$ instead of $s_{1}^{2}$ and $s_{2}^{2}$, and set the degrees of freedom to $df = n_{1} + n_{2} - 2$. In this case, the standard error (for $\bar{X}_{12}$) and $t$-score are calculated as follows:

$$
\begin{array}{l}
SE_{1 2} = \sqrt {s_{p}^{2} / n_{1} + s_{p}^{2} /n_{2}} = s_{p} \sqrt {1 / n_{1} + 1 /n_{2}}, \\
t = \frac {\bar {x}_{1 2}}{s_{p} \sqrt {1 / n_{1} + 1 / n_{2}}}, \\
\end{array}
$$

where $\bar{x}_{1 2}$ is the difference between the observed sample means as before.

The code in Example 4 below shows how to perform a two sample $t$-test with pooled variance. For comparison, the same data used in Example 3 (male and female body temperatures) is used below.

## Example 4: Two-Sample t-Test with Pooled Variance

The code in the cell below performs a two-sample $t$-test using pooled variance, comparing the body temperatures of male and female subjects. This is essentially a repetition of Example 3.

In [ ]:
# @title Example 4: Two-Sample t-Test with Pooled Variance

import pandas as pd
import numpy as np
from scipy import stats

# Set seed
np.random.seed(42)  # For reproducible results

# Define variables
dataset = bodyTemp_df
col_name = "Gender"
x1 = "M"
x2 = "F"
sample_var = "Temperature"


# Set parameters
sample_1 = dataset[dataset[col_name] == x1]
sample_2 = dataset[dataset[col_name] == x2]

# Compute statistics --------------------------------
# number
n1 = len(sample_1)
n2 = len(sample_2)

# means
mean1 = sample_1[sample_var].mean()
mean2 = sample_2[sample_var].mean()

# standard deviations
s1 = sample_1[sample_var].std()
s2 = sample_2[sample_var].std()

# Calculate mean difference
mean_diff = mean1 - mean2

# Calculate pooled variance (assuming equal population variances)
pooled_variance = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)

# Calculate standard error of the difference using pooled variance
se_diff = np.sqrt(pooled_variance * (1/n1 + 1/n2))

# Calculate degrees of freedom for pooled t-test
df = n1 + n2 - 2

# Calculate t-score
t_score = mean_diff / se_diff

# Calculate p-value based on alternative hypothesis H_A: mu_12 != 0
p_value = 2 * (1 - stats.t.cdf(abs(t_score), df))

# Display results
print("\nTwo-Sample t-Test Results (Pooled Variance)")
print("=" * 40)
print(f"Sample sizes: n1 = {n1}, n2 = {n2}")
print(f"Sample means: x̄₁ = {mean1:.2f}, x̄₂ = {mean2:.2f}")
print(f"Sample standard deviations: s₁ = {s1:.2f}, s₂ = {s2:.2f}")
print(f"Pooled variance: s²_pooled = {pooled_variance:.2f}")
print(f"Standard error: SE₁₂ = {se_diff:.2f}")
print(f"Degrees of freedom: df = {df:.0f}")
print(f"t-score: t = {t_score:.2f}")
print(f"p-value: p_obs = {p_value:.2f}")
print("-" * 40)

If the code is correct, you should see the following output:
```text
Two-Sample t-Test Results (Pooled Variance)
========================================
Sample sizes: n1 = 49, n2 = 51
Sample means: x̄₁ = 98.20, x̄₂ = 98.46
Sample standard deviations: s₁ = 1.05, s₂ = 0.85
Pooled variance: s²_pooled = 0.91
Standard error: SE₁₂ = 0.19
Degrees of freedom: df = 98
t-score: t = -1.36
p-value: p_obs = 0.18
----------------------------------------
```

With this particular dataset, there is no significant difference in the output using pooled variance compared to the results obtained above in Example 3.

## **Exercise 4: Two-Sample t-Test with Pooled Variance**

In the cell below, write the code to perform a two-sample $t$-test with pooled variance using the same data that you used above in **Exercise 3**.

In [ ]:
# @title Exercise 4: Two-Sample t-Test with Pooled Variance



If your code is correct, you should see the following output:
```text
Two-Sample t-Test Results (Pooled Variance)
========================================
Sample sizes: n1 = 9, n2 = 8
Sample means: x̄₁ = 26.33, x̄₂ = 18.12
Sample standard deviations: s₁ = 9.77, s₂ = 4.09
Pooled variance: s²_pooled = 58.73
Standard error: SE₁₂ = 3.72
Degrees of freedom: df = 15
t-score: t = 2.20
p-value: p_obs = 0.04
```

Using pooled variance has some effect on the results of the two-Sample $t$-Test. With pooled variance, the standard error (SE₁₂) increased from $3.56$ to $3.72$, the degrees of freedom ($df$) increased from $10.07$, and the $t$-score _decreased_ from $2.30$ to $2.20$. While the $p$-value was also effected, the change was too small to see the difference with 2-significant figures.    

 ### **_8.2.3 Paired t-test_**

When using two-sample _t_-test to investigate the relationship between a binary variable that defines the grouping of the individuals (e.g., gender) and the response variable (e.g., body temperature), we hope that the individuals in the two samples are quite comparable except for the characteristic that defines the groups. In our body temperature example, the two groups (female and male) should be similar with respect to other possibly important factors affecting body temperature, such as age and ethnicity, so the only factor that separates the two groups is gender. This way, if the observed difference in mean body temperature between the two groups is significant, it is likely to be related to gender. (Of course, even if we establish that there is a relationship between gender and body temperature, we cannot define it as causation since the data are obtained from an observational study.)

While we hope that the two samples taken from the population are comparable except for the characteristic that defines the grouping, this is not guaranteed in general. For the body temperature example, one group might include relatively older participants. To mitigate the influence of other important factors (e.g., age) that are not the focus of our study, we sometimes **pair** each individual in one group with an individual in the other group so that the paired individuals are very similar to each other except for the characteristic that defines the grouping. For example, when we are investigating the relationship between gender and body temperature and we are concerned that the two samples might not be comparable with respect to factors such as age and ethnicity, we can recruit twins with different genders for our study so that each individual in the female group is paired by her twin in the male group. This way, we make sure that the two samples are exactly the same in terms of age and ethnicity, and they are comparable with respect to other possibly important characteristics such as genetic factors.

Often, not only are the two samples related, they in fact include the same individuals. For example, suppose that we are investigating the effect of a specific diet on blood pressure. We could of course recruit a sample of subjects and ask them to follow that specific diet for six months. For comparison, we recruit another sample of subjects who do not follow our prescribed diet. At the end of the study period, we compare the two groups in terms of their blood pressure using the two-sample t-test described previously. However, it is possible that just by chance the subjects in the diet group tend to have lower blood pressure even before our experiment starts. For example, they might be relatively younger than the control group, or they might exercise more. To avoid such issues, we can design our experiment so that the same individuals participate in both groups. To this end, we can recruit subjects that are not following our prescribed diet, measure their blood pressure, ask them to follow the diet for six months, and measure their blood pressure again at the end of the study. This way, the two groups include the same subjects under different conditions: before the diet and after the diet.

The two-sample $t$-test we described previously is based on the assumption that the two groups are unrelated (independent). When the individuals in the two groups are paired, we use the paired $t$-test to take the pairing of the observations between the two groups into account. In what follows, we use the study of the effect of tobacco smoke on platelet function by Levine [16] to describe this method. In his study, Levine hypothesized that the higher frequency of arterial thrombosis in cigarette smokers could be partially explained by increased platelet aggregation caused by smoking. To test this hypothesis, Levine conducted an experiment where he selected a group of eleven people and measured platelet aggregation before and after smoking a cigarette for each individual. Therefore, observations in the "Before" sample and "After" sample are from the same subjects. For each subject, an observation in the "Before" sample is paired with an observation in the "After" sample.

--------------

>To account for the dependency between the observations in the two groups, we use the **paired $t$-test** instead of the independent two-sample $t$-test discussed above. Specifically, we compare each observation in the first group to its corresponding observation in the second group. Using the difference between the paired observations, the hypothesis testing problem reduces to a single sample $t$-test problem (Sect. 7.4).

--------------

For the `platelet` data, we want to compare platelet aggregation measurements for the same person before and after smoking. Let us define a new random variable $D$, which represents the difference in platelet aggregation from before to after. Because we believe that platelet aggregation before smoking tends to be less than after, we expect $D$ to be negative on average. Therefore, we could express the alternative hypothesis as $H_{A}:\mu<0$, where $\mu$ is the population average of the random variable $D$. However, to be conservative, we consider the possibility that $\mu$ could also be positive and specify the alternative hypothesis as $H_{A}:\mu\neq0$. Then the null hypothesis is that the mean of change in platelet aggregation due to smoking is zero, $H_{0}:\mu=0$. We can use the methods discussed in the previous chapter for inference regarding one population mean to find confidence intervals for $ \mu$ and test the null hypothesis. As before, we use the sample mean, $\bar{D}$, for this purpose.

Using the `platelet` data, the observed value of $\bar{D}$ is $\bar{d} = -10.27$. Using the sample standard deviation $s = 7.98$ for $D$ and the sample size $n = 11$, the standard error (i.e., estimated standard deviation) of $\bar{D}$ is
$$
SE = \frac {7.98}{\sqrt {11}} = 2.41.
$$

Suppose that we are interested in 95% confidence interval estimate for $\mu$ (i.e., population mean of the difference between before and after smoking). Using the $t$-distribution with $n-1=10$ degrees of freedom, we obtain $t_{\mathrm{crit}}=2.23$. Therefore, the 95% confidence interval is

$$
\begin{array}{l} [\bar{d}-t_{\mathrm {crit}} \times SE, \bar{d}+t_{\mathrm {crit}} \times SE] = [-10.27 - 2.23 \times 2.41, -10.27 + 2.23 \times 2.41] \\
\hspace{5.7cm}= [-15.64, -4.90]. \ \end{array}
$$

At 0.$95$ confidence level, we believe that the true mean of the difference in platelet aggregation measurements before and after smoking is between $-15.64$ and $-4.90$. Note that this range includes negative values only. More specifically, it does not include the value $0$ specified by the null hypothesis.

To perform hypothesis testing, we find the $t$-score (here, $\mu_{0}=0$ according to the null hypothesis) as follows:
$$
t = \frac {- 10.27}{2.41} = - 4.26.
$$

To find the $p$-value, we find the upper tail probability of $|-4.26|=4.26$ from the $t$-distribution with 10 degrees of freedom, and multiply the results by 2 for two-sided hypothesis testing:
$$
p_{\mathrm {obs}} = 2P(T > 4. 2 6) = 2 \times 0.0008 = 0.0016.
$$

At $0.01$ confidence level, we can reject the null hypothesis and conclude that the test result is statistically significant. In this case, we interpret this as a statistically significant relationship between smoking and platelet aggregation.

The Python code for performing this particular paired $t$-test is shown below in Example 5.

## **Exercise 5: Paired t-test**

The code in the cell below performs a paired $t$-test on the `platelet` data that was previously stored in the DataFrame `platelet_df` at the start of this lesson.

In [ ]:
# @title Exercise 5: Paired t-Test

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Set seed
np.random.seed(42)  # For reproducible results


# Calculate differences (D = Before - After)
platelet_df['D'] = platelet_df['Before'] - platelet_df['After']

print("Platelet Aggregation Data (Before vs After Smoking)")
print("=" * 50)
print(platelet_df)

# Calculate sample statistics
n = len(platelet_df)
mean_D = platelet_df['D'].mean()
std_D = platelet_df['D'].std()

print(f"\nSample Statistics:")
print(f"Number of subjects: n = {n}")
print(f"Mean difference: D̄ = {mean_D:.2f}")
print(f"Standard deviation of differences: s_D = {std_D:.2f}")

# Calculate standard error of the mean difference
se_D = std_D / np.sqrt(n)

# Calculate t-statistic for paired t-test
# H₀: μ = 0 (no change in platelet aggregation)
# Hₐ: μ ≠ 0 (there is a change)
t_stat = mean_D / se_D

# Degrees of freedom for paired t-test
df = n - 1

# Calculate p-value (two-tailed test)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))

# Calculate 95% confidence interval for μ
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha/2, df)
margin_of_error = t_critical * se_D
ci_lower = mean_D - margin_of_error
ci_upper = mean_D + margin_of_error

# Display results
print(f"\nPaired t-Test Results")
print("=" * 30)
print(f"Sample size: n = {n}")
print(f"Mean difference: D̄ = {mean_D:.2f}")
print(f"Standard deviation of differences: s_D = {std_D:.2f}")
print(f"Standard error: SE_D = {se_D:.2f}")
print(f"Degrees of freedom: df = {df}")
print(f"t-statistic: t = {t_stat:.2f}")
print(f"p-value: p_obs = {p_value:.4f}")

If the code is correct, you should see the following output:
```text
Platelet Aggregation Data (Before vs After Smoking)
==================================================
    Before  After   D
0       25     27  -2
1       25     29  -4
2       27     37 -10
3       44     56 -12
4       30     46 -16
5       67     82 -15
6       53     57  -4
7       53     80 -27
8       52     61  -9
9       60     59   1
10      28     43 -15

Sample Statistics:
Number of subjects: n = 11
Mean difference: D̄ = -10.27
Standard deviation of differences: s_D = 7.98

Paired t-Test Results
==============================
Sample size: n = 11
Mean difference: D̄ = -10.27
Standard deviation of differences: s_D = 7.98
Standard error: SE_D = 2.40
Degrees of freedom: df = 10
t-statistic: t = -4.27
p-value: p_obs = 0.0016

```

With a $p$-value = $0.0016$, we can reject the null hypothesis $H_{0} = 0$ at the $0.01$ confidence level and conclude that the test result is statistically significant. In this case, we interpret this as a statistically significant relationship between smoking and platelet aggregation.

Now suppose that we had ignored the pairing and treated the two groups `Before` and `After` as unrelated (independent). Then we would erroneously conduct a two-sample $t$-test. The value of the $t$-score in this case would be $t = -1.42$, the degrees of freedom would be $df = 19.52$, and the resulting $p$-value for two-sided hypothesis testing would be $p_{obs} = 0.17$. Then, we would fail to reject the null hypothesis at commonly used significant levels and conclude that the relationship between smoking and platelet aggregation is not statistically significant.

-----------------------

>When performing $t$-tests, ignoring the dependence between the two groups is inappropriate and possibly results in the wrong conclusion.

---------------------

In general, we perform the paired $t$-test as follows. Suppose that there are $n$ observations in the first sample and n observations in the second sample. Therefore, there are n pairs of observations and 2n observations in total. Now consider the _i_ th pair of observations, $ x_{i1} $ and $ x_{i2} $ , where $ x_{i1} $ is the observation in the first sample, and $ x_{i2} $ is the corresponding observation in the second sample. We find the difference $ d_{i}=x_{i1}-x_{i2} $ between the paired observations. We assume that $ d_{i} $ is an observation for the random variable D. We will have n observed values for D, where each value is the difference between a pair of observations from the original data. We now use the single sample t-test (Sect. 7.4) to evaluate the null hypothesis $ H_{0}:\mu=\mu_{0} $ , where $ \mu $ is the population mean of D, and $ \mu_{0} $ is usually zero (i.e., the difference between paired observations is zero on average). As before, the alternative is either one sided or two sided.

Using the observed sample mean of $D$, which we denote as $\bar{d}$, and the observed sample standard deviation $s$, we find confidence intervals for $\mu$:

$$
[ \bar {d} - t_{\mathrm {crit}} \times SE, \bar {d} + t_{\mathrm {crit}} \times SE ],
$$

where $t_{crit}$ is the factor obtained for the desired confidence level $c$ from the $t$-distribution with $n - 1$ degrees of freedom, and $ SE = s / \sqrt{n}$ is the standard error for the statistic $\bar{D}$ .

To test the null hypothesis $ H_{0}:\mu=0 $ , we calculate the $T$ statistic,

$$
T = \frac {\bar {D}}{S / \sqrt {n}},
$$

where $\bar{D}$ is the sample mean of the paired differences, $S$ is the sample standard deviation of $D$, and $n$ is the number of pairs. If the null hypothesis is true, then the test statistic $T$ has the $t$-distribution with $n - 1$ degrees of freedom. We calculate the corresponding t-score as follows:

$$
t = \frac {\bar{d}}{s / \sqrt {n}}.
$$

Then the $p$-value is the probability of having as extreme or more extreme values than the observed t-score:

$$
\begin{array}{l}
\text {if } H_{A}: \mu > 0, \quad p_{\mathrm {obs}} = P(T \geq t), \\
\text {if } H_{A}: \mu < 0, \quad p_{\mathrm {obs}} = P(T \leq t), \\
\text {if } H_{A}: \mu \neq 0, \quad p_{\mathrm {obs}} = 2 \times P (T \geq | t |). \\
\end{array}
$$

Instead of creating the variable $D$ and performing the single-sample $t$-test, we can use Python to perform a paired $t$-test directly as shown in Example 6. The results shown in the output are identical to those found by the single sample $t$-test for the difference ($D$).

## **Exercise 6: Direct Paired $t$-Test**

The code in the cell below performs a direct paired $t$-test on the `platelet` dataset without creating a difference variable $D$ by subtracting the `Before` from the `After` measurements.

In [ ]:
# @title Exercise 6: Direct Paired  t -Test

import numpy as np
from scipy import stats

# Perform paired t-test directly
t_statistic, p_value = stats.ttest_rel(platelet_df['Before'], platelet_df['After'])

# Alternative hypothesis: two-sided (default)
# This means we're testing if there's a difference (either positive or negative)
print("Paired t-test Results:")
print(f"t-statistic: {t_statistic:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Sample size: {len(platelet_df['Before'])}")

# Interpretation
alpha = 0.05  # significance level
if p_value < alpha:
    print("Result: Significant difference between before and after scores (reject null hypothesis)")
else:
    print("Result: No significant difference between before and after scores (fail to reject null hypothesis)")

If the code is correct, you should see the following output:
```text
Paired t-test Results:
t-statistic: -4.2716
p-value: 0.0016
Sample size: 11
Result: Significant difference between before and after scores (reject null hypothesis)
```

## **8.3 Inference about the Relationship Between Two Binary Variables**

In this section, we discuss statistical inference methods for evaluating the relationship between two binary random variables. As an example, suppose that we want to investigate whether smoking during pregnancy increases the risk of having a low birthweight baby. We use the `birthwt` data set for this purpose. The random variable of interest (i.e., response variable) is `low`, indicating whether the baby's birthweight was less than $2.5$ kg. The explanatory variable is `smoke`, indicating the mother's smoking status during pregnancy.

A common way to analyze the relationship between binary (in general, categorical) variables is to use contingency tables. Contingency tables are a tabular representations of the frequencies for all possible combinations of the two variables. To obtain the contingency table in R-Commander, click Statistics → Contingency tables → Two-way table. Select smoke as the Row variable (i.e., X) and low as the Column variable (i.e., $Y$), as in Fig. 8.9. Check Row percentages and uncheck Chi-square test of independence for now. The resulting tables shown in the Output window (see Tables 8.1 and 8.2). Table 8.1 provides the frequency of each cell. The first row in this table shows non-smoking mothers, and the second row shows smoking mothers. The first column shows the number of babies of normal birthweight, and the second column shows the number of babies of low birthweight. We regard nonsmoking mothers as the first group, with $n_{1}=115$, and smoking mothers as the second group. with $n_{2}=74$.

To perform statistical inference in this section, we rely on the CLT and assume that the distributions of sample proportions (i.e., sample means for $n$ binary random variables) are approximately normal. For this assumption to be reasonable, the frequencies in each cell of the contingency table should be at least $5$.

We are interested in the relationship between the two binary variables. If having low-birthweight babies is related to smoking during pregnancy, we expect the population means of low-birthweight babies to be different between smoking and nonsmoking mothers. Of course, for binary random variables, population mean is the same as the population proportion for the outcome of interest (denoted as $1$). We denote the population proportion of low-birthweight babies for nonsmoking mothers as $\mu_{1}$, and the population proportion of low-birthweight babies for smoking mothers as $\mu_{2}$. We use $\mu_{1 2}$ to denote the difference between these two proportions: $\mu_{1 2} = \mu_{1} - \mu_{2}$.

If smoking and low birthweight are related, we expect the two population proportions to be different and $\mu_{1 2}$ to be away from zero. Therefore, we can express our hypothesis regarding the relationship between the two variables as $H_{A}:\mu_{1 2}\neq0$. We could of course be more specific and specify our hypothesis as $H_{A}:\mu_{12}<0$ if we believe that the population proportion of low-birthweight babies among nonsmoking mothers, $\mu_{1}$, is less than the population proportion of low-birthweight babies among smoking mothers, $\mu_{2}$.
However, to be conservative, we use the two-sided alternative. The corresponding null hypothesis is then $H_{0}:\mu_{12}=0$.

To find the confidence intervals for $\mu_{1 2}$, we use the difference between sample proportions, $\bar{X}_{1 2} = \bar{X}_{1} - \bar{X}_{2} $, as the statistic. According to the CLT, the sampling distribution of $\bar{X}_{1 2}$ is approximately normal,

$$
\bar {X}_{1 2} \sim N \left(\mu_{1} - \mu_{2}, \sigma_{1}^{2} / n_{1} + \sigma_{2}^{2} / n_{2}\right).
$$

As before, we use $SD_{1 2}$ to denote the standard deviation of the sampling distribution of $\bar{X}_{1 2}$. Recall that for binary random variables, the population variance is $\sigma^{2} = \mu(1 - \mu)$. Therefore, we can write $SD_{1 2}$ as follows:

$$
SD_{1 2} = \sqrt {\mu_{1} \left(1 - \mu_{1}\right) / n_{1} + \mu_{2} \left(1 - \mu_{2}\right) / n_{2}}.
$$

Then, we write the sample distribution of $\bar{X}_{1 2}$ as

$$
\bar {X}_{1 2} \sim N \left(\mu_{1 2}, SD_{1 2}^{2}\right).
$$

The observed value of this statistic for the sample data is $\bar{x}_{1 2} = \bar{x}{1} - \bar{x}{2}$. For binary random variables, it is common to use $p_{1}$ and $p_{2}$ for observed sample proportions. Therefore, we denote the observed difference between sample proportions as $p_{1 2}$. For our example, $p_{1} = 0.25$ and $p_{2} = 0.40 $. These are the point estimates for $\mu_{1}$ and $\mu_{2}$, respectively. As a result, the point estimate of $\mu_{12}$ is $p_{12} = 0.25 - 0.40 = -0.15$ .

We also use $p_{1}$ and $p_{2}$ to estimate the standard deviation $SD_{1 2}$. We refer to our estimate of $SD_{12}$ as the standard error and denote it as $SE_{1 2}$:

$$
SE_{1 2} = \sqrt {p_{1} \left(1 - p_{1}\right) / n_{1} + p_{2} \left(1 - p_{2}\right) / n_{2}}.
$$

For the above example,
$$
SE_{1 2} = \sqrt {0.25 (1 - 0.25) / 115 + 0.40 (1 - 0.40) / 74} = 0.07.
$$

Using the point estimate $p_{1 2}$ along with the standard error $SE_{1 2}$, we can find confidence intervals for $\mu_{1 2}$ as follows:

$$
\left[p_{1 2} - z_{\mathrm {crit}} \times SE_{1 2}, p_{1 2} + z_{\mathrm {crit}} \times SE_{1 2} \right],
$$

where $z_{crit}$ is obtained for a given confidence level $c$ as before. Note that we use $z_{crit}$ even though the population variances were unknown. This is because we did not use the data to estimate them separately; rather, we used our point estimates for the population proportions.

For the birthweight example, the 95% confidence interval of $\mu_{12}$ is

$$
[- 0.15 - 2 \times 0.07, -0.15 + 2 \times 0.07] = [- 0.29, - 0.01].
$$

Therefore, we are $95$% confident that the difference between the two population proportions falls between $-0.29$ and $-0.01$. Note that all the values in this range are negative. The value of $\mu_{1 2}$ is negative when $\mu_{1}$ (i.e., population proportion of low-birthweight babies among nonsmoking mothers) is less than $\mu_{2}$ (i.e., population proportion of low-birthweight babies among smoking mothers). More specifically, the interval does not include $0$, which is the value specified by the null hypothesis.

More formally, we test the null hypothesis, $H_{0}:\mu_{12}=0$, using the two-sample $z$-test as follows. First, we obtain the $z$-score,

$$
z = \frac {\bar {p}_{1 2}}{SE_{1 2}}.
$$

Then, we calculate the $p$-value, which is the observed significance level:

$$
\begin{array}{l}
\text {if } H_{A}: \mu_{1 2} > 0, \quad p_{\mathrm {obs}} = P (Z \geq z), \\
\text {if } H_{A}: \mu_{1 2} < 0, \quad p_{\mathrm {obs}} = P (Z \leq z), \\
\text {if } H_{A}: \mu_{1 2} \neq 0, \quad p_{\mathrm {obs}} = 2 \times P (Z \geq | z |). \\
\end{array}
$$

The above tail probabilities are obtained from the standard normal distribution. For our example,
$$
z = \frac {-0.15}{0.07} = -2.14.
$$

Because we specified the alternative hypothesis as $H_{1 2}:\mu_{1 2}\neq0$, the $p$-value is two times the upper tail probability of $|-2.14|=2.14$ from the standard normal distribution, that is, $p_{obs}=2\times0.016=0.032$.

At $0.05$ level (but not at $0.01$ level), we can reject the null hypothesis and conclude that the observed difference in the proportion of low-birthweight babies is statistically significant and is probably not due to the chance alone. Therefore, at $0.05$ level, we can conclude that the two variables, smoking during pregnancy and having low-birthweight babies, are related.

## **Exercise 7: Inference about Two Binary Variables**

The code in the cell below, creates a contingency table for maternal smoking (`smoke`) versus low birth weight (`low`).

In [ ]:
# @title Exercise 7: Inference about Two Binary Variables

import pandas as pd
import numpy as np

# Define variables
dataset = birthwt_df
rand_var1 = "smoke"
rand_var2 = "low"

# Create the contingency table
contingency_table = pd.crosstab(dataset[rand_var1], dataset[rand_var2],
                                   margins=True,
                                   margins_name="Total")


# Show Table 8.1
print(f"Table: 8.1 Contingency table of {rand_var1} by {rand_var2}\n")
print(contingency_table)
print("\n")

# Show Table 8.2
column_percentages = contingency_table.div(contingency_table.loc[:, 'Total'], axis=0) * 100
print(f"Table 8.2 Sample proportions of {rand_var1} by {rand_var2}\n")
print(column_percentages.round(3))
print()

The output shows Tables 8.1 and 8.2. **Table 8.1** provides the frequency of each cell. The first row in this table shows non-smoking mothers, and the second row shows smoking mothers. The first column shows the number of babies of normal birthweight, and the second column shows the number of babies of low birthweight. We regard nonsmoking mothers as the first group, with $n_{1}=115 $, and smoking mothers as the second group. with $n_{2}=74 $.

To perform statistical inference in this section, we rely on the CLT and assume that the distributions of sample proportions (i.e., sample means for n binary random variables) are approximately normal. For this assumption to be reasonable, the frequencies in each cell of the contingency table should be at least 5.

**Table 8.2** (row percentages) provides the relative frequencies or sample proportions for each row separately. For example, the proportion of babies with low birthweight among nonsmoking mothers (i.e., first row and second column) is $0.25$.

We are interested in the relationship between the two binary variables. If having low-birthweight babies is related to smoking during pregnancy, we expect the population means of low-birthweight babies to be different between smoking and nonsmoking mothers. Of course, for binary random variables, population mean is the same as the population proportion for the outcome of interest (denoted as 1). We denote the population proportion of low-birthweight babies for nonsmoking mothers as $\mu_{1}$, and the population proportion of low-birthweight babies for smoking mothers as $ \mu_{2}$. We use $\mu_{12}$ to denote the difference between these two proportions: $\mu_{12} = \mu_{1} - \mu_{2}$.

If smoking and low birthweight are related, we expect the two population proportions to be different and $ \mu_{12} $ to be away from zero. Therefore, we can express our hypothesis regarding the relationship between the two variables as $H_{A}:\mu_{12}\neq0 $. We could of course be more specific and specify our hypothesis as $H_{A}:\mu_{12}<0$ if we believe that the population proportion of low-birthweight babies among nonsmoking mothers, $\mu_{1}$, is less than the population proportion of low-birthweight babies among smoking mothers, $\mu_{2}$. However, to be conservative, we use the two-sided alternative. The corresponding null hypothesis is then $H_{0}:\mu_{12}=0$.

To find the confidence intervals for $ \mu_{12} $, we use the difference between sample proportions, $ \bar{X}{12} = \bar{X}{1} - \bar{X}{2} $ , as the statistic. According to the CLT, the sampling distribution of $ \bar{X}{12} $ is approximately normal,
$$
\bar {X} _ {1 2} \sim N \left(\mu_{1}-\mu_{2}, \sigma_ {1}^{2}/n_{1} + \sigma_{2}^{2}/ n _ {2}\right).
$$

As before, we use $SD_{12}$ to denote the standard deviation of the sampling distribution of $ \bar{X}{12}$. Recall that for binary random variables, the population variance is $ \sigma^{2} = \mu(1 - \mu) $. Therefore, we can write $ SD{12} $ as follows:
$$ S D _ {1 2} = \sqrt {\mu_ {1} \left(1 - \mu_ {1}\right) / n _ {1} + \mu_ {2} \left(1 - \mu_ {2}\right) / n _ {2}}. $$

Then, we write the sample distribution of $\bar{X}_{12}$ as

$$
\bar {X} _ {1 2} \sim N \left(\mu_ {1 2}, S D _ {1 2}^{2}\right).
$$

The observed value of this statistic for the sample data is $ \bar{x}{12} = \bar{x}{1}-\bar{x}{2}$. For binary random variables, it is common to use $p_{1}$ and $ p_{2} $ for observed sample proportions. Therefore, we denote the observed difference between sample proportions as $ p_{12} $. For our example, $ p_{1} = 0.25 $ and $ p_{2} = 0.40 $. These are the point estimates for $\mu_{1}$ and $\mu_{2}$, respectively. As a result, the point estimate of $\mu_{12}$ is $p_{12} = 0.25 - 0.40 = -0.15$.

We also use $p_{1}$ and $p_{2}$ to estimate the standard deviation $ SD_{12}$. We refer to our estimate of $SD_{12}$ as the standard error and denote it as $ SE_{12}$:

$$
SE_{12} = \sqrt {p_{1} \left(1-p_{1}\right)/n_{1}+p_{2} \left(1-p_{2}\right)/n_{2}}.
$$

For the above example,
$$
SE_{12} = \sqrt {0.25 (1-0.25) / 115 + 0.40 (1-0. 40)/74} = 0.07.
$$

Using the point estimate $p_{12}$ along with the standard error $SE_{12}$, we can find confidence intervals for $\mu_{12}$ as follows:

$$
\left[p_{12}-z_ {\mathrm {crit}} \times SE_{12}, p_{12} + z_{\mathrm {crit}} \times SE_{12} \right],
$$

where $z_{crit}$ is obtained for a given confidence level $c$ as before. Note that we use $ z_{crit} $ even though the population variances were unknown. This is because we did not use the data to estimate them separately; rather, we used our point estimates for the population proportions.

For the birthweight example, the 95% confidence interval of $\mu_{12}$ is

$$
[-0.15 - 2 \times 0.07, -0.15 + 2\times 0.07 ] = [ -0.29, - 0.01].
$$

Therefore, we are $95$% confident that the difference between the two population proportions falls between $-0.29$ and $-0.01$. Note that all the values in this range are negative. The value of $\mu_{12}$ is negative when $\mu_{1} $(i.e., population proportion of low-birthweight babies among nonsmoking mothers) is less than $\mu_{2}$ (i.e., population proportion of low-birthweight babies among smoking mothers). More specifically, the interval does not include 0, which is the value specified by the null hypothesis.

More formally, we test the null hypothesis, $H_{0}:\mu_{12}=0 $, using the two-sample $z$-test as follows. First, we obtain the z-score,
$$ z = \frac {\bar {p} _ {1 2}}{S E _ {1 2}}. $$
Then, we calculate the p-value, which is the observed significance level:
$$
\begin{array}{l}
\text {if }H_{A}: \mu_ {12} > 0, \quad p_{\mathrm {obs}} = P (Z \geq z), \\
\text {if } H_{A}: \mu_ {12} < 0, \quad p _ {\mathrm {obs}} = P (Z \leq z), \\
\text {if } H _ {A}: \mu_ {12} \neq 0, \quad p _ {\mathrm {obs}} = 2 \times P(Z \geq |z|). \\
 \end{array}
$$

The above tail probabilities are obtained from the standard normal distribution.

For our example,
$$
z = \frac {-0.15}{0.07} = -2.14.
$$

Because we specified the alternative hypothesis as $ H_{12}:\mu_{12}\neq0$, the $p$-value is two times the upper tail probability of $|-2.14|=2.14$ from the standard normal distribution, that is, $p_{obs}=2\times0.016=0.032$.

At $0.05$ level (but not at $0.01$ level), we can reject the null hypothesis and conclude that the observed difference in the proportion of low-birthweight babies is statistically significant and is probably not due to the chance alone. Therefore, at $0.05$ level, we can conclude that the two variables, smoking during pregnancy and having low-birthweight babies, are related.

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()